# httpbin Deployment 会话亲和

本 Notebook 创建一个共享 httpbin Tool 和三个独立 Deployment，对比全部会话亲和模式。亲和配置不能通过 `deployment update` 修改，因此每个模式使用独立 Deployment。

| 模式 | 目标实例不可用时 | 实例所有权 |
| --- | --- | --- |
| `BEST_EFFORT` | 可以选择其他实例 | 共享 |
| `STRICT` | 请求失败，不迁移 | 共享 |
| `EXCLUSIVE` | 不迁移 | 每个 affinity ID 独占一个实例 |

请求与响应都使用 `X-Httpbin-Affinity`。本例通过响应 header 和 HTTP 状态观察路由契约，不开启真实 hostname，也不使用验证脚本。请把 `AGR_ROLE_ARN` 替换为允许 AGR 拉取目标 CCR 镜像的 CAM 角色 ARN。

In [ ]:
%env AGR_REGION=ap-shanghai
%env AGR_DOMAIN=tencentags.com
%env AGR_ROLE_ARN=qcs::cam::uin/replace-me:roleName/replace-me
%env HTTPBIN_TOOL_NAME=httpbin-affinity-your-name
%env BEST_EFFORT_DEPLOYMENT_NAME=httpbin-best-effort-your-name
%env STRICT_DEPLOYMENT_NAME=httpbin-strict-your-name
%env EXCLUSIVE_DEPLOYMENT_NAME=httpbin-exclusive-your-name
!agr status

## 1. 创建共享 Tool

先把四个名称中的 `your-name` 改为同一个唯一后缀。

In [ ]:
!agr tool create \
  --region "$AGR_REGION" \
  --tool-name "$HTTPBIN_TOOL_NAME" \
  --tool-type custom \
  --persistent \
  --role-arn "$AGR_ROLE_ARN" \
  --network-configuration '{"NetworkMode":"PUBLIC"}' \
  --custom-configuration '{"Image":"ccr.ccs.tencentyun.com/ags.dev/go-httpbin:v2.25.0","ImageRegistryType":"personal","Command":["/bin/go-httpbin"],"Args":["-host","0.0.0.0","-port","8080"],"Env":[{"Name":"EXCLUDE_HEADERS","Value":"X-Access-Token"}],"Ports":[{"Name":"http","Port":8080,"Protocol":"TCP"}],"Resources":{"CPU":"200m","Memory":"500Mi"},"Probe":{"HttpGet":{"Path":"/status/200","Port":8080,"Scheme":"HTTP"},"ReadyTimeoutMs":30000,"ProbeTimeoutMs":1000,"ProbePeriodMs":3000,"SuccessThreshold":1,"FailureThreshold":10}}' \
  --wait

## 2. 创建三个 Deployment

复制 `ToolId`。三个 Deployment 使用相同 header 名和 30 秒 `STOP` 空闲策略，便于手工观察目标实例不可用后的差异。`EXCLUSIVE` 最多允许三个独占实例。

In [ ]:
%env HTTPBIN_TOOL_ID=sdt-replace-me
!agr deployment create --region "$AGR_REGION" --deployment-name "$BEST_EFFORT_DEPLOYMENT_NAME" --tool-id "$HTTPBIN_TOOL_ID" --scaling-configuration '{"MinInstanceCount":0,"MaxInstanceCount":2,"MaxInstanceRequestConcurrency":10}' --lifecycle-configuration '{"IdleTimeoutSeconds":30,"IdleAction":"STOP"}' --affinity-configuration '{"Mode":"BEST_EFFORT","HeaderName":"X-Httpbin-Affinity"}'
!agr deployment create --region "$AGR_REGION" --deployment-name "$STRICT_DEPLOYMENT_NAME" --tool-id "$HTTPBIN_TOOL_ID" --scaling-configuration '{"MinInstanceCount":0,"MaxInstanceCount":2,"MaxInstanceRequestConcurrency":10}' --lifecycle-configuration '{"IdleTimeoutSeconds":30,"IdleAction":"STOP"}' --affinity-configuration '{"Mode":"STRICT","HeaderName":"X-Httpbin-Affinity"}'
!agr deployment create --region "$AGR_REGION" --deployment-name "$EXCLUSIVE_DEPLOYMENT_NAME" --tool-id "$HTTPBIN_TOOL_ID" --scaling-configuration '{"MinInstanceCount":0,"MaxInstanceCount":3,"MaxInstanceRequestConcurrency":1}' --lifecycle-configuration '{"IdleTimeoutSeconds":30,"IdleAction":"STOP"}' --affinity-configuration '{"Mode":"EXCLUSIVE","HeaderName":"X-Httpbin-Affinity"}'

## 3. 获取三个 Deployment Token

按创建顺序复制三个 `DeploymentId`。分别获取 Token 后，再把每个响应的 `Data.Response.Response.Token` 复制到下一单元。Token 不能跨 Deployment 使用。

In [ ]:
%env BEST_EFFORT_DEPLOYMENT_ID=dpl-replace-me
%env STRICT_DEPLOYMENT_ID=dpl-replace-me
%env EXCLUSIVE_DEPLOYMENT_ID=dpl-replace-me
!agr api call AcquireDeploymentToken --region "$AGR_REGION" --request '{"DeploymentId":"'$BEST_EFFORT_DEPLOYMENT_ID'"}' --output json
!agr api call AcquireDeploymentToken --region "$AGR_REGION" --request '{"DeploymentId":"'$STRICT_DEPLOYMENT_ID'"}' --output json
!agr api call AcquireDeploymentToken --region "$AGR_REGION" --request '{"DeploymentId":"'$EXCLUSIVE_DEPLOYMENT_ID'"}' --output json

In [ ]:
%env BEST_EFFORT_TOKEN=dpt-replace-me
%env STRICT_TOKEN=dpt-replace-me
%env EXCLUSIVE_TOKEN=dpt-replace-me

## 4. `BEST_EFFORT`：优先复用，允许迁移

第一次请求不带 affinity header。复制响应中的 `X-Httpbin-Affinity` 值并设置到第二个单元；带回该值时，Deployment 优先选择此前实例。若保持空闲至少 30 秒使实例停止，再带回相同 ID，请求仍可选择其他实例继续执行。

In [ ]:
!curl --include --silent --show-error --header "X-Access-Token: $BEST_EFFORT_TOKEN" "https://8080-$BEST_EFFORT_DEPLOYMENT_ID.$AGR_REGION.agents.$AGR_DOMAIN/headers"

In [ ]:
%env BEST_EFFORT_AFFINITY_ID=replace-with-response-header
!curl --include --silent --show-error --header "X-Access-Token: $BEST_EFFORT_TOKEN" --header "X-Httpbin-Affinity: $BEST_EFFORT_AFFINITY_ID" "https://8080-$BEST_EFFORT_DEPLOYMENT_ID.$AGR_REGION.agents.$AGR_DOMAIN/headers"

## 5. `STRICT`：必须复用，不允许迁移

先获取并复制响应 affinity ID，再携带它请求。随后保持空闲至少 30 秒，使目标实例停止；再次携带同一个 ID 时，`STRICT` 应返回失败而不是选择新实例。使用 `--include` 保留 HTTP 状态和错误响应。

In [ ]:
!curl --include --silent --show-error --header "X-Access-Token: $STRICT_TOKEN" "https://8080-$STRICT_DEPLOYMENT_ID.$AGR_REGION.agents.$AGR_DOMAIN/headers"

In [ ]:
%env STRICT_AFFINITY_ID=replace-with-response-header
!curl --include --silent --show-error --header "X-Access-Token: $STRICT_TOKEN" --header "X-Httpbin-Affinity: $STRICT_AFFINITY_ID" "https://8080-$STRICT_DEPLOYMENT_ID.$AGR_REGION.agents.$AGR_DOMAIN/headers"

## 6. `EXCLUSIVE`：一个 affinity ID 独占一个实例

连续发送两次不带 affinity header 的请求，分别复制两个响应 affinity ID。两个 ID 对应两个互不共享、不可迁移的实例；后续请求必须带回对应 ID。实例上限也因此限制了可同时存在的独占会话数。

In [ ]:
!curl --include --silent --show-error --header "X-Access-Token: $EXCLUSIVE_TOKEN" "https://8080-$EXCLUSIVE_DEPLOYMENT_ID.$AGR_REGION.agents.$AGR_DOMAIN/headers"
!curl --include --silent --show-error --header "X-Access-Token: $EXCLUSIVE_TOKEN" "https://8080-$EXCLUSIVE_DEPLOYMENT_ID.$AGR_REGION.agents.$AGR_DOMAIN/headers"

In [ ]:
%env EXCLUSIVE_AFFINITY_ID_A=replace-with-first-response-header
%env EXCLUSIVE_AFFINITY_ID_B=replace-with-second-response-header
!curl --include --silent --show-error --header "X-Access-Token: $EXCLUSIVE_TOKEN" --header "X-Httpbin-Affinity: $EXCLUSIVE_AFFINITY_ID_A" "https://8080-$EXCLUSIVE_DEPLOYMENT_ID.$AGR_REGION.agents.$AGR_DOMAIN/headers"
!curl --include --silent --show-error --header "X-Access-Token: $EXCLUSIVE_TOKEN" --header "X-Httpbin-Affinity: $EXCLUSIVE_AFFINITY_ID_B" "https://8080-$EXCLUSIVE_DEPLOYMENT_ID.$AGR_REGION.agents.$AGR_DOMAIN/headers"

## 7. 清理资源

先删除全部 Deployment；确认删除完成后，再删除共享 Tool。

In [ ]:
!agr deployment delete "$BEST_EFFORT_DEPLOYMENT_ID" --region "$AGR_REGION"
!agr deployment delete "$STRICT_DEPLOYMENT_ID" --region "$AGR_REGION"
!agr deployment delete "$EXCLUSIVE_DEPLOYMENT_ID" --region "$AGR_REGION"
!agr instance list --tool-id "$HTTPBIN_TOOL_ID" --region "$AGR_REGION"

若仍有非 `STOPPED` 实例，逐个复制实例 ID，在新单元先执行 `%env HTTPBIN_INSTANCE_ID=replace-me`，再执行 `!agr instance delete "$HTTPBIN_INSTANCE_ID" --region "$AGR_REGION" --yes --wait`。全部处理后删除共享 Tool。

In [ ]:
!agr tool delete "$HTTPBIN_TOOL_ID" --region "$AGR_REGION" --yes --wait